In [ ]:
# =============================================================================
# 12e — Does DAL restore class discriminativeness
#
# Background. HA training collapsed the MEL and NV attention maps onto each
# other. Pearson went from -0.275 at alpha=0 to +0.968 at alpha=5.
# DAL minimises pairwise cosine between class maps and was designed to
# prevent exactly this.
#
# Caveat. DAL acts on TOKEN projections. HA acts on ATTENTION.
# Different tensors. DAL reaches attention only through shared weights.
# This notebook measures both so the distinction is visible.
#
# Success criteria
#   attn_class_pearson  0.968  ->  toward 0
#   MEL union AUC       0.920  ->  hold above 0.85
#   test AUC-ROC        ~0.96  ->  hold above 0.93
# =============================================================================
from pathlib import Path
import sys, json, re
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib as mpl


def find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "generate_finer_cam_panderm.py").exists():
            return p.resolve()
    raise FileNotFoundError


REPO = find_repo_root()
for extra in [REPO, REPO / "external" / "PanDerm" / "classification"]:
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from src.eval.cam_eval_utils import (
    CAM_GRID, norm01, mask_to_cam_grid, resize_centercrop_mask)

HAM_ROOT  = REPO / "data" / "HAM10000"
DAL_DIR   = REPO / "external" / "checkpoints_dal"
EVAL_ROOT = REPO / "outputs" / "mel_nv" / "eval_cams" / "gap"
QC_CSV    = REPO / "results" / "annotation_qc" / "annotation_qc_manifest.csv"
OUT_DIR   = REPO / "results" / "dal"
FIG       = OUT_DIR / "figures"
FIG.mkdir(parents=True, exist_ok=True)

POOLING = "mean"
BLOCK   = -1
CLASSES = ["MEL", "NV"]
MEL_IDX, NV_IDX = 0, 1
TOP_K   = 20

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")

mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300,
                     "savefig.bbox": "tight", "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
print("device:", DEVICE)

In [ ]:
# =============================================================================
# Discover DAL checkpoints from the flat naming convention.
#   checkpoint-best-gap-ha<HA>-dal<DAL>.pth      p means decimal point
#   dal_logs/<same stem>.log.txt                 training log if present
#
# Reference is the original alpha=5 model with no extra epochs.
# The ha5-dal0p1 run acts as a near control for the extra 10 epochs.
# =============================================================================
import re

CKPT_DIR = REPO / "external" / "checkpoints5"
LOG_DIR  = CKPT_DIR / "dal_logs"

PAT = re.compile(r"checkpoint-best-gap-ha([\d p]+?)-dal([\dp]+)\.pth$")


def unp(s):
    return float(s.replace("p", "."))


runs = {
    "ref_ha5": {
        "path": CKPT_DIR / "checkpoint-best-gap-ha5.pth",
        "ha": 5.0, "dal": 0.0, "kind": "reference", "log": None,
    }
}

for f in sorted(CKPT_DIR.glob("checkpoint-best-gap-ha*-dal*.pth")):
    m = PAT.search(f.name)
    if not m:
        print(f"  unparsed: {f.name}")
        continue
    ha, dal = unp(m.group(1)), unp(m.group(2))
    lg = LOG_DIR / f"{f.stem}.log.txt"
    runs[f"ha{ha:g}_dal{dal:g}"] = {
        "path": f, "ha": ha, "dal": dal,
        "kind": "dal_only" if ha == 0 else "ha_dal",
        "log": lg if lg.exists() else None,
    }

info = pd.DataFrame([
    {"run": k, "ha": v["ha"], "dal": v["dal"], "kind": v["kind"],
     "log": v["log"] is not None, "exists": v["path"].exists()}
    for k, v in runs.items()
]).sort_values(["ha", "dal"])

print(info.to_string(index=False))
print(f"\n{len(runs)} checkpoints, {int(info.log.sum())} with logs")

missing = info[~info.exists]
if len(missing):
    raise FileNotFoundError(f"missing checkpoints:\n{missing.to_string(index=False)}")

In [ ]:
# =============================================================================
# If the evaluate_ha logging patch was active, everything is already here.
# =============================================================================
rows = []
for k, v in runs.items():
    lg = v.get("log")
    if lg is None or not lg.exists():
        continue
    last = json.loads(lg.read_text().strip().split("\n")[-1])
    rows.append({
        "run": k, "ha": v["ha"], "dal": v["dal"],
        **{c: last.get(f"val_{c}") for c in [
            "auc_roc", "balanced_accuracy", "ha_dice",
            "inside_minus_area_baseline",
            "attn_class_pearson", "attn_class_cosine", "token_class_cosine"]},
        "train_dal_loss": last.get("train_dal_loss"),
        "train_ha_loss": last.get("train_ha_loss"),
    })

logs = pd.DataFrame(rows).sort_values(["ha", "dal"])
PATCH_WAS_LIVE = "attn_class_pearson" in logs and logs.attn_class_pearson.notna().any()

print(f"logging patch active during training: {PATCH_WAS_LIVE}\n")
print(logs.round(4).to_string(index=False))

if PATCH_WAS_LIVE:
    print("\nThe answer is already here. Cells 4 to 6 confirm it on the")
    print("annotated subset. Cell 7 plots either way.")
else:
    print("\nPatch was not active. Cells 4 to 6 will regenerate CAMs.")

In [ ]:
from run_class_finetuning_ha import (
    build_attention_gradcam_map, _unwrap_model, build_all_cls_patch_maps,
    dal_cosine_loss, unpack_model_outputs)
from scripts.generate_finer_cam_panderm import (
    build_panderm_model, remap_official_finetune_checkpoint_keys,
    extract_checkpoint_state_dict, infer_panderm_variant_from_state_dict,
    infer_checkpoint_pooling)
from models.builder import get_eval_transforms

PREPROCESS = get_eval_transforms(which_img_norm="imagenet",
                                 img_resize=256, center_crop=True)
_cache = {}


def load_model(key):
    if key in _cache:
        return _cache[key]
    p = runs[key]["path"]
    ck = torch.load(p, map_location="cpu", weights_only=False)
    raw, _ = extract_checkpoint_state_dict(ck)

    got = infer_checkpoint_pooling(ck, raw)
    if got is not None and got != POOLING:
        raise RuntimeError(f"{key}: checkpoint pooling {got}, expected {POOLING}")

    state = remap_official_finetune_checkpoint_keys(raw)
    variant = infer_panderm_variant_from_state_dict(state)
    m = build_panderm_model(num_classes=2, variant=variant,
                            use_mean_pooling=(POOLING == "mean"))
    miss, unexp = m.load_state_dict(state, strict=False)
    bad = [k for k in list(miss) + list(unexp)
           if k.startswith(("norm.", "fc_norm.", "head."))]
    if bad:
        raise RuntimeError(f"{key}: critical keys {bad}")

    m.to(DEVICE).eval()
    for q in m.parameters():
        q.requires_grad_(True)
    _cache[key] = m
    return m


def load_tensor(rel):
    img = Image.open((HAM_ROOT / str(rel)).resolve()).convert("RGB")
    return PREPROCESS(img).unsqueeze(0).to(DEVICE)


def both_class_cams(model, x):
    """Return cam_mel, cam_nv, probs, token_cosine. One forward."""
    base = _unwrap_model(model)
    if hasattr(base, "clear_xai_state"):
        base.clear_xai_state()

    with torch.enable_grad():
        out = model(x, return_patch_tokens=True, store_attn=True, attn_layer=BLOCK)
        logits, patch_tokens = unpack_model_outputs(out)

        cams = []
        for idx in (MEL_IDX, NV_IDX):
            t = torch.full((logits.shape[0],), idx,
                           device=logits.device, dtype=torch.long)
            cams.append(build_attention_gradcam_map(
                model, logits, t, create_graph=False).detach())

        with torch.no_grad():
            tok = float(dal_cosine_loss(
                build_all_cls_patch_maps(model, patch_tokens)).item())
            probs = torch.softmax(logits.detach(), 1)[0].float().cpu().numpy()

    if hasattr(base, "clear_xai_state"):
        base.clear_xai_state()
    return (cams[0][0].float().cpu().numpy(),
            cams[1][0].float().cpu().numpy(), probs, tok)

In [ ]:
# =============================================================================
# For each checkpoint, on the 34 annotated images.
#   attention pearson and cosine between the MEL and NV maps
#   token cosine, DAL's own objective
#   MEL union AUC for the target class map
# =============================================================================
from sklearn.metrics import roc_auc_score

eval_df = pd.read_csv(EVAL_ROOT / "eval_images.csv")
qcp = pd.read_csv(QC_CSV).query("qc_pass")


def mel_union_grid(image_id):
    sub = qcp[(qcp.Image_ID == image_id) & (qcp.label_group == "MEL")]
    if not len(sub):
        return None
    acc = None
    for _, r in sub.iterrows():
        m = np.array(Image.open(r.mask_path).convert("L")) > 127
        acc = m if acc is None else (acc | m)
    return mask_to_cam_grid(resize_centercrop_mask(acc)[0],
                            already_cropped=True) >= 0.5


unions = {i: mel_union_grid(i) for i in eval_df.image_id}


def topk_fixed(c, k=TOP_K):
    f = np.asarray(c, np.float32).ravel()
    o = np.zeros(f.size, bool)
    o[np.argpartition(-f, k - 1)[:k]] = True
    return o.reshape(c.shape)


rows = []
for key, meta in runs.items():
    model = load_model(key)
    for _, r in eval_df.iterrows():
        x = load_tensor(r.image_rel_path)
        cm, cn, probs, tok = both_class_cams(model, x)

        a, b = norm01(cm).ravel(), norm01(cn).ravel()
        tgt = cm if r.gt_label == "MEL" else cn
        un = unions.get(r.image_id)

        rows.append({
            "run": key, "ha": meta["ha"], "dal": meta["dal"],
            "image_id": r.image_id, "gt_label": r.gt_label,
            "pearson": float(np.corrcoef(a, b)[0, 1]),
            "cosine": float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)),
            "topk_iou": float((topk_fixed(norm01(cm)) & topk_fixed(norm01(cn))).sum()
                              / ((topk_fixed(norm01(cm)) | topk_fixed(norm01(cn))).sum() + 1e-8)),
            "token_cosine": tok,
            "p_mel": float(probs[MEL_IDX]),
            "pred": CLASSES[int(np.argmax(probs))],
            "mel_auc": (float(roc_auc_score(un.ravel(), norm01(tgt).ravel()))
                        if un is not None and 0 < un.sum() < un.size else np.nan),
        })
    print(f"done {key}")

per = pd.DataFrame(rows)
per.to_csv(OUT_DIR / "dal_per_image.csv", index=False)
print(f"\n{len(per)} rows")

In [ ]:
def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if len(x) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    b = [np.median(rng.choice(x, len(x), replace=True)) for _ in range(n)]
    return float(np.median(x)), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


summ = []
for key, g in per.groupby("run"):
    pm, plo, phi = boot_ci(g.pearson)
    am, alo, ahi = boot_ci(g.mel_auc)
    summ.append({
        "run": key, "ha": g.ha.iloc[0], "dal": g.dal.iloc[0],
        "pearson": pm, "pearson_ci": f"[{plo:+.3f}, {phi:+.3f}]",
        "cosine": g.cosine.median(),
        "topk_iou": g.topk_iou.median(),
        "token_cosine": g.token_cosine.median(),
        "mel_auc": am, "mel_auc_ci": f"[{alo:.3f}, {ahi:.3f}]",
        "subset_acc": (g.gt_label == g.pred).mean(),
    })

s = pd.DataFrame(summ).sort_values(["ha", "dal"])
s = s.merge(logs[["run", "auc_roc", "balanced_accuracy"]], on="run", how="left")
s.to_csv(OUT_DIR / "dal_summary.csv", index=False)

print("=" * 100)
print("DOES DAL RESTORE CLASS DISCRIMINATIVENESS")
print("=" * 100)
print(s.round(3).to_string(index=False))

ref = s[s.run == "ref_ha5"].iloc[0]
print(f"\nreference at DAL=0: pearson {ref.pearson:+.3f}  "
      f"MEL AUC {ref.mel_auc:.3f}")

ok = s[(s.dal > 0) & (s.ha > 0) & (s.pearson < 0.5) & (s.mel_auc > 0.85)]
if len(ok):
    print(f"\nDAL values restoring separability while holding alignment:")
    print(ok[["run", "dal", "pearson", "mel_auc"]].round(3).to_string(index=False))
else:
    print("\nNo DAL value drops pearson below 0.5 while keeping MEL AUC above 0.85.")

In [ ]:
# =============================================================================
# FIGURE. Four panels against DAL lambda.
#   1  attention pearson, the collapse metric
#   2  token cosine, DAL's own objective
#   3  MEL union AUC, does alignment survive
#   4  test AUC-ROC, does accuracy survive
#
# READ
#   panel 1 falls and panel 3 holds  ->  DAL works
#   panel 2 falls but panel 1 flat   ->  decoupled, token and attention
#                                        carry class info independently
#   panel 2 flat                     ->  DAL never optimised, check lambda
# =============================================================================
h = s[(s.ha > 0)].sort_values("dal")
d_only = s[s.ha == 0]

fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
specs = [
    ("pearson", "MEL vs NV map correlation", (-0.4, 1.05), 0.0),
    ("token_cosine", "token map cosine, DAL target", None, None),
    ("mel_auc", "MEL union AUC", (0.4, 1.0), 0.85),
    ("auc_roc", "test AUC-ROC", (0.8, 1.0), 0.93),
]

for a, (col, title, ylim, thresh) in zip(ax, specs):
    if col not in h or h[col].isna().all():
        a.text(0.5, 0.5, "not available", ha="center", transform=a.transAxes)
        a.set_title(title)
        continue
    a.plot(h.dal, h[col], "o-", color="#4682B4", lw=2, ms=6, label="HA=5 + DAL")
    if len(d_only) and not pd.isna(d_only[col].iloc[0]):
        a.axhline(d_only[col].iloc[0], color="#FF8C00", ls="--", lw=1.5,
                  label="DAL only")
    if thresh is not None:
        a.axhline(thresh, color="k", ls=":", lw=1)
    a.set_xlabel(r"DAL $\lambda$")
    a.set_title(title, fontsize=9)
    if ylim:
        a.set_ylim(*ylim)

ax[0].legend(frameon=False, fontsize=8)
fig.suptitle("Does DAL restore class discriminativeness lost to alignment training",
             fontsize=11, y=1.03)
fig.savefig(FIG / "fig7_dal_sweep.png")
plt.show()